# 05 — Synthetic 3-rater vs Examiner-Grounded GT: a comparative study

**Claim under test**: examiner-grounded prior-art labels (SIRP, KIPO) and synthetic 3-rater
expert-matching labels (Park 2026a, ExpDataSet) measure *different and complementary*
signals of relevance in semiconductor expert/prior-art retrieval.

**Inputs**:
- `data/patents/prior_art_pairs.parquet` — 7,500 examiner-grounded pairs (this term, SIRP)
- `data/experts/curated_ratings.parquet` — 7,800 synthetic 3-rater ratings (Park 2026a)
- `data/experts/curated_ratings_pivot.parquet` — 2,600 subjects pivoted to rater_1/2/3

**Outputs of this notebook** (run after `make ratings`):
- Label-distribution comparison
- Inter-rater agreement on the synthetic side (κ, ICC) — already in `reliability_report.md`
- The two label sources cover non-overlapping decision surfaces — argued from coverage and from semantics.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent  # notebook run from notebooks/
print('repo root:', ROOT)

PAIRS = ROOT / 'data' / 'patents' / 'prior_art_pairs.parquet'
RATINGS = ROOT / 'data' / 'experts' / 'curated_ratings.parquet'
PIVOT = ROOT / 'data' / 'experts' / 'curated_ratings_pivot.parquet'
RELIABILITY = ROOT / 'data' / 'experts' / 'reliability_report.json'

pairs = pd.read_parquet(PAIRS)
ratings = pd.read_parquet(RATINGS)
pivot = pd.read_parquet(PIVOT)
rel = json.loads(RELIABILITY.read_text())
print('shapes:', pairs.shape, ratings.shape, pivot.shape)

## 1. Label space and scale

In [ ]:
print('--- Examiner-grounded (SIRP) ---')
print('label distribution:')
print(pairs['label'].value_counts())
print('difficulty distribution:')
print(pairs['difficulty'].value_counts())
print()
print('--- Synthetic 3-rater (curated) ---')
print('relevance_score distribution:')
print(ratings['relevance_score'].value_counts().sort_index())
print()
print('per (problem, expert) subject mean score quantiles:')
print(pivot['mean_score'].quantile([0.10, 0.25, 0.50, 0.75, 0.90]))

## 2. Inter-rater reliability (synthetic only — examiner GT has single rater)

From `reliability_report.json`:
- Cohen's κ pairwise rater_1 vs 2, 1 vs 3, 2 vs 3
- Fleiss' κ across 3 raters
- ICC(2,1)

In [ ]:
for k, v in rel.items():
    if k not in ('outputs', 'raters_named'):
        print(f'{k}: {v}')
print()
print('Interpretation (Landis & Koch 1977 + Koo & Li 2016):')
fleiss = rel['fleiss_kappa']
icc = rel['icc_2_1']
print(f'  Fleiss κ = {fleiss:.3f} ' + ('— fair (0.21–0.40)' if 0.21 <= fleiss <= 0.40 else 'see scale'))
print(f'  ICC(2,1) = {icc:.3f} ' + ('— moderate (0.50–0.75)' if 0.50 <= icc <= 0.75 else 'see scale'))

## 3. Semantic coverage — they measure different things

**Examiner-grounded (SIRP)**: relevance := "this prior-art document is the substantive
ground for rejecting this patent application". A 1-vs-0 binary objectivable signal
issued by a public official (KIPO examiner).

**Synthetic 3-rater (curated)**: relevance := "this expert is well-matched to solve this
SME-stated semiconductor problem". A subjective fitness signal aggregated across 3
named raters.

These do not interchange; **they are complementary axes** of relevance for the same
AFCP-EM platform — Expert track uses curated, PriorArt track uses examiner-grounded.

## 4. Use-case partition (lock the experimental contract)

In [ ]:
summary = pd.DataFrame([
    {
        'track': 'AFCP-EM-Expert',
        'query': 'SME technical problem',
        'candidate': 'curated expert profile (100)',
        'GT source': 'synthetic 3-rater (Park 2026a)',
        'rows': len(ratings),
        'subjects': len(pivot),
        'eval metrics': 'MRR, NDCG@5, P@5, leakage_rate (Tier-1/2/3)',
    },
    {
        'track': 'AFCP-EM-PriorArt',
        'query': 'rejected patent application (SIRP)',
        'candidate': 'in-corpus prior-art patent (773 + IPC peers)',
        'GT source': 'KIPO examiner citation (this term)',
        'rows': len(pairs),
        'subjects': int(pairs['target_patent_id'].nunique()),
        'eval metrics': 'MRR, NDCG@5, Recall@K, leakage_rate (multi-jurisdiction)',
    },
])
summary

## 5. What the comparison enables

1. **External validity check**: any AFCP-EM design choice tested on curated 3-rater data
   can be re-validated on examiner-grounded labels (and vice versa) — this is what is
   missing from V3.3.5 ([Park 2026a]) and is the new contribution this term.
2. **Crowd-noise estimate**: Fleiss κ ≈ 0.26 and ICC(2,1) ≈ 0.55 set an empirical
   ceiling on agreement under simulated raters — useful as a published benchmark for
   follow-on work on synthetic-label calibration.
3. **Bridge in §5.5**: leakage-rate metrics defined identically on both tracks (see
   `docs/leakage_protocol.md`), letting the compliance gate's structural effect be
   reported across a real-label and a synthetic-label dataset.